# Notebook 08 — Full Evaluation — Three Scenarios and Benchmark Comparison

Implements the complete evaluation framework required by the research project, covering three distinct evaluation scenarios that demonstrate system flexibility, cross-source generalisation testing, baseline comparison, and benchmark comparison against five IEEE/ACM reference papers.

## What This Notebook Does

### Scenario 1 — Train/Test Split Ratios
Tests the diverse fusion system across three split configurations:
- 70/15/15 (standard split used throughout project)
- 80/10/10 (more training data)
- 60/20/20 (larger test set)
Demonstrates system stability — all splits perform within 0.0022 Macro F1.

### Scenario 2 — Ablation Study
Compares four system configurations to quantify each component's contribution:
- Zero-shot + XGBoost (no fine-tuning)
- Stylometric only (no LLM)
- ModernBERT only (no stylometric features)
- Full Fusion (complete proposed system)
Proves each component adds genuine value to the final system.

### Scenario 3 — Dataset Composition
Compares three dataset configurations to assess training data diversity impact:
- Original single source (Qwen2.5 only)
- Cleaned single source (salutations removed)
- Diverse multi-source (Qwen2.5 + Gemini BEC)
Key finding: internal performance appears similar but external BEC detection goes from 2.7% to 99.8%, proving diversity is the critical factor.

### External BEC Generalisation Test
Tests all system configurations on 4,181 Gemini Flash 2.5 BEC emails:
- Original and cleaned systems: all 4,181 emails are fully unseen
- Diverse system: 2,091 held-out emails (2,090 used in training)
This distinction is important for correct interpretation of results.

### Benchmark Comparison
Compares our system against five post-2022 IEEE/ACM papers:
- Chanis & Arampatzis (ACM 2024) — F1: 0.9843 — best comparable benchmark
- Zhang et al. (IEEE 2025) — F1: 0.9756
- Heiding et al. (IEEE 2024) — Accuracy: >0.95
- Afane et al. (IEEE 2024) — Recall: 0.9895 (drops to 0.9316 on rephrased)
- Salloum et al. (IEEE 2022) — foundational reference

## Inputs
- data/processed/stylometric_features_diverse.csv (from Notebook 09)
- data/processed/modernbert_diverse_features.csv (from Notebook 05b)
- data/processed/stylometric_features_final.csv (from Notebook 02/04)
- data/processed/modernbert_features.csv (from Notebook 05b)
- data/processed/stylometric_features_cleaned.csv (from Notebook 09)
- data/processed/modernbert_cleaned_features.csv (from Notebook 05b)
- data/processed/bec_test_holdout.csv (from Notebook 09)
- data/processed/bec_diverse_modernbert.csv (from Notebook 05b)
- models/xgboost_modernbert.pkl (from Notebook 06)
- models/xgboost_diverse.pkl (from Notebook 09)

## Outputs
- results/scenario1_split_ratios.csv and .png
- results/scenario2_ablation.csv and .png
- results/scenario3_dataset_composition.csv and .png
- results/all_scenarios_heatmap.png
- results/benchmark_comparison.csv and .png

## Important Notes on Results Interpretation
- Support values for diverse system test set: (600, 600, 687, 1887)
- BEC detection 99.8% is on holdout of partially seen source — not fully unseen
- Benchmark comparison limited by inconsistent metric reporting across papers
- All experiments use random_state=42 for full reproducibility

## Runtime
Approximately 5-10 minutes

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             precision_score, recall_score)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import time
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Load all feature sources
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
modernbert = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")
llm = pd.read_csv(DATA_PROCESSED / "llm_features.csv")
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")

print("All data loaded:")
print(f"  Stylometric: {stylo.shape}")
print(f"  ModernBERT:  {modernbert.shape}")
print(f"  Zero-shot:   {llm.shape}")
print(f"  Dataset:     {dataset.shape}")

# Build feature matrices
stylo_features = stylo.drop(columns=['label'])
modernbert_scores = modernbert[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
y = stylo['label']

# Main fusion feature matrix (ModernBERT + Stylometric)
X_fusion = pd.concat([stylo_features, modernbert_scores], axis=1)

print(f"\nFusion feature matrix: {X_fusion.shape}")

In [ ]:
#reusable metrics function
def compute_all_metrics(y_true, y_pred, y_proba=None, model_name="Model"):
    """Compute all 6 evaluation metrics for a model."""
    
    metrics = {}
    metrics['model'] = model_name
    
    # 1. Macro F1
    metrics['macro_f1'] = f1_score(y_true, y_pred, average='macro')
    
    # 2. MCC
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    
    # 3. AUC-ROC (if probabilities available)
    if y_proba is not None:
        y_true_bin = label_binarize(y_true, classes=[0,1,2])
        metrics['auc_roc'] = roc_auc_score(y_true_bin, y_proba, 
                                            multi_class='ovr', average='macro')
    else:
        metrics['auc_roc'] = np.nan
    
    # 4. False Positive Rate (legitimate misclassified as phishing)
    cm = confusion_matrix(y_true, y_pred)
    fp_legit = cm[0][1] + cm[0][2]  # legitimate predicted as phishing
    tn_legit = cm[0][0]
    total_legit = cm[0].sum()
    metrics['fpr'] = (fp_legit) / total_legit if total_legit > 0 else 0
    
    # 5. AI Phishing Recall
    metrics['ai_recall'] = recall_score(y_true, y_pred, labels=[2], average='macro')
    
    return metrics

In [ ]:
#training all systems and building comparison tables
# Same split for everything
def get_splits(X, y):
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y)
    X_v, X_te, y_v, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)
    return X_tr, X_te, y_tr, y_te

def train_xgb(X_train, y_train):
    cc = y_train.value_counts().sort_index()
    total = len(y_train)
    cw = {cls: total/(len(cc)*count) for cls, count in cc.items()}
    sw = y_train.map(cw)
    m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=42, eval_metric='mlogloss', verbosity=0)
    m.fit(X_train, y_train, sample_weight=sw)
    return m

all_results = []

# System 1 — Fusion (ModernBERT + Stylometric)
X_tr, X_te, y_tr, y_te = get_splits(X_fusion, y)
m_fusion = train_xgb(X_tr, y_tr)
preds = m_fusion.predict(X_te)
proba = m_fusion.predict_proba(X_te)
all_results.append(compute_all_metrics(y_te, preds, proba, "Fusion (ModernBERT+Stylo)"))

# System 2 — ModernBERT + Stylometric via zero-shot comparison
X_zs = pd.concat([stylo_features, 
                  llm[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]], axis=1)
X_tr, X_te, y_tr, y_te = get_splits(X_zs, y)
m_zs = train_xgb(X_tr, y_tr)
preds = m_zs.predict(X_te)
proba = m_zs.predict_proba(X_te)
all_results.append(compute_all_metrics(y_te, preds, proba, "Fusion (Zero-shot+Stylo)"))

# System 3 — Stylometric only
X_tr, X_te, y_tr, y_te = get_splits(stylo_features, y)
m_stylo = train_xgb(X_tr, y_tr)
preds = m_stylo.predict(X_te)
proba = m_stylo.predict_proba(X_te)
all_results.append(compute_all_metrics(y_te, preds, proba, "Stylometric only"))

# System 4 — ModernBERT standalone
X_tr, X_te, y_tr, y_te = get_splits(modernbert_scores, y)
preds = modernbert['llm_label'].iloc[y_te.index]
proba = modernbert_scores.iloc[y_te.index].values
all_results.append(compute_all_metrics(y_te, preds, proba, "ModernBERT standalone"))

# System 5 — Zero-shot standalone
X_tr, X_te, y_tr, y_te = get_splits(
    llm[['conf_legitimate','conf_human_phishing','conf_ai_phishing']], y)
preds = llm['llm_label'].map({'legitimate':0,'human_phishing':1,
                               'ai_generated_phishing':2}).iloc[y_te.index]
all_results.append(compute_all_metrics(y_te, preds, None, "Zero-shot standalone"))

# Build comparison table
comparison = pd.DataFrame(all_results)
comparison = comparison[['model','macro_f1','auc_roc','mcc','fpr','ai_recall']]
comparison.columns = ['System','Macro F1','AUC-ROC','MCC','FPR','AI Recall']

print("FULL SYSTEM COMPARISON \n")
print(comparison.to_string(index=False))

# Save
comparison.to_csv(RESULTS_DIR / "system_comparison.csv", index=False)
print("\nSaved to system_comparison.csv")

In [ ]:
# Load the external BEC dataset (CSV format)
external_path = BASE_DIR / "data" / "raw" / "external_test" / "bec_dataset.csv"

bec = pd.read_csv(external_path)
print(f"BEC dataset loaded: {bec.shape}")
print(f"\nColumns: {bec.columns.tolist()}")
print(f"\nFirst 2 rows:")
print(bec.head(2))
print(f"\nLabel values:")
print(bec['label'].value_counts())

In [ ]:
from pathlib import Path

external_dir = BASE_DIR / "data" / "raw" / "external_test"

print(f"Looking in: {external_dir}")
print(f"Folder exists: {external_dir.exists()}\n")

if external_dir.exists():
    print("Files found:")
    for f in external_dir.iterdir():
        print(f"  {f.name}")
else:
    print("Folder doesn't exist. Searching whole project for Excel files...")
    for f in BASE_DIR.rglob("*.xlsx"):
        print(f"  Found: {f}")
    for f in BASE_DIR.rglob("*.xls"):
        print(f"  Found: {f}")

In [ ]:
# Combine subject and body into full email text (matching our training format)
bec['text'] = bec['subject'].astype(str) + " " + bec['body'].astype(str)

# Drop rows with missing labels
bec = bec.dropna(subset=['label'])
bec['label'] = bec['label'].astype(int)

# These are all AI-generated phishing — in our system that's class 2
# But we test whether the system flags them as ANY phishing (1 or 2)
print(f"BEC emails ready: {len(bec)}")
print(f"Average email length: {bec['text'].str.len().mean():.0f} chars")
print(f"\nSample email:")
print(bec['text'].iloc[0][:300])

In [ ]:
import nltk, spacy, textstat, string, re
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nlp = spacy.load("en_core_web_sm")
from tqdm import tqdm

# Paste the same four feature functions from Notebook 02/04
def extract_basic_features(text):
    text = str(text); words = text.split()
    sentences = nltk.sent_tokenize(text)
    num_words = len(words) if len(words)>0 else 1
    num_sentences = len(sentences) if len(sentences)>0 else 1
    return {'email_length':len(text),'num_words':len(words),'num_sentences':num_sentences,
        'num_paragraphs':len([p for p in text.split('\n\n') if p.strip()]),
        'avg_word_length':np.mean([len(w) for w in words]) if words else 0,
        'max_word_length':max([len(w) for w in words]) if words else 0,
        'avg_sentence_length':num_words/num_sentences,'unique_words':len(set(words)),
        'type_token_ratio':len(set(words))/num_words,
        'vocab_richness':len(set(w.lower() for w in words))/num_words,
        'num_exclamations':text.count('!'),'num_questions':text.count('?'),
        'num_commas':text.count(','),'num_periods':text.count('.'),
        'exclamation_ratio':text.count('!')/num_words,'question_ratio':text.count('?')/num_words,
        'num_capitals':sum(1 for c in text if c.isupper()),
        'capital_ratio':sum(1 for c in text if c.isupper())/len(text) if text else 0,
        'num_special_chars':sum(1 for c in text if c in string.punctuation),
        'special_char_ratio':sum(1 for c in text if c in string.punctuation)/len(text) if text else 0}

def extract_phishing_features(text):
    text=str(text); words=text.split(); num_words=len(words) if len(words)>0 else 1
    tl=text.lower()
    urls=re.compile(r'http[s]?://(?:[a-zA-Z0-9$-_@.&+!*\\(\\),]|(?:%[0-9a-fA-F]{2}))+').findall(text)
    uw=['urgent','immediately','expire','suspend','verify','confirm','update','click','login',
        'password','account','bank','limited','offer','winner','prize','free','congratulations',
        'selected','act now']
    gw=['dear','hello','hi','greetings','good morning','good afternoon','dear customer','dear user']
    twords=['suspended','terminated','blocked','restricted','unauthorized','illegal','fraud','risk']
    return {'num_urls':len(urls),'has_url':int(len(urls)>0),'url_ratio':len(urls)/num_words,
        'num_http':tl.count('http://'),'num_https':tl.count('https://'),
        'urgency_word_count':sum(1 for w in uw if w in tl),
        'threat_word_count':sum(1 for w in twords if w in tl),
        'has_greeting':int(any(g in tl for g in gw)),'has_unsubscribe':int('unsubscribe' in tl),
        'has_dear':int('dear' in tl),'has_winner':int('winner' in tl or 'won' in tl),
        'has_free':int('free' in tl),'has_click_here':int('click here' in tl),
        'has_verify':int('verify' in tl or 'verification' in tl),'has_account':int('account' in tl),
        'has_password':int('password' in tl),'has_bank':int('bank' in tl),
        'has_invoice':int('invoice' in tl or 'payment' in tl),
        'num_digits':sum(c.isdigit() for c in text),
        'digit_ratio':sum(c.isdigit() for c in text)/len(text) if text else 0}

def extract_readability_features(text):
    text=str(text)
    if len(text.split())<10:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade','gunning_fog',
            'smog_index','coleman_liau_index','automated_readability_index',
            'dale_chall_readability','difficult_words','linsear_write_formula','text_standard']}
    try:
        return {'flesch_reading_ease':textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade':textstat.flesch_kincaid_grade(text),
            'gunning_fog':textstat.gunning_fog(text),'smog_index':textstat.smog_index(text),
            'coleman_liau_index':textstat.coleman_liau_index(text),
            'automated_readability_index':textstat.automated_readability_index(text),
            'dale_chall_readability':textstat.dale_chall_readability_score(text),
            'difficult_words':textstat.difficult_words(text),
            'linsear_write_formula':textstat.linsear_write_formula(text),
            'text_standard':float(str(textstat.text_standard(text,float_output=True)))}
    except:
        return {k:0 for k in ['flesch_reading_ease','flesch_kincaid_grade','gunning_fog',
            'smog_index','coleman_liau_index','automated_readability_index',
            'dale_chall_readability','difficult_words','linsear_write_formula','text_standard']}

def extract_syntactic_features(text):
    text=str(text)
    if len(text)>5000: text=text[:5000]
    doc=nlp(text); tt=len(doc) if len(doc)>0 else 1
    pc={}
    for token in doc: pc[token.pos_]=pc.get(token.pos_,0)+1
    return {'noun_ratio':pc.get('NOUN',0)/tt,'verb_ratio':pc.get('VERB',0)/tt,
        'adj_ratio':pc.get('ADJ',0)/tt,'adv_ratio':pc.get('ADV',0)/tt,
        'pronoun_ratio':pc.get('PRON',0)/tt,'propn_ratio':pc.get('PROPN',0)/tt,
        'det_ratio':pc.get('DET',0)/tt,'punct_ratio':pc.get('PUNCT',0)/tt,
        'num_ratio':pc.get('NUM',0)/tt,'num_entities':len(doc.ents),
        'entity_ratio':len(doc.ents)/tt,
        'stopword_ratio':sum(1 for t in doc if t.is_stop)/tt,
        'unique_punct':len(set(t.text for t in doc if t.is_punct))}

def extract_all_features(text):
    f={}; f.update(extract_basic_features(text)); f.update(extract_phishing_features(text))
    f.update(extract_readability_features(text)); f.update(extract_syntactic_features(text))
    return f

print("Extracting stylometric features for BEC emails")
bec_features = []
for idx, row in tqdm(bec.iterrows(), total=len(bec)):
    try:
        feat = extract_all_features(row['text'])
        bec_features.append(feat)
    except:
        continue

bec_stylo = pd.DataFrame(bec_features)
print(f"\n Features extracted: {bec_stylo.shape}")
bec_stylo.to_csv(DATA_PROCESSED / "bec_stylometric.csv", index=False)
print("Saved bec_stylometric.csv")

In [ ]:
#external generalisation test
# Load BEC features and scores
bec_raw = pd.read_csv(BASE_DIR / "data" / "raw" / "external_test" / "bec_dataset.csv")
bec_raw = bec_raw.dropna(subset=['label']).reset_index(drop=True)
bec_stylo = pd.read_csv(DATA_PROCESSED / "bec_stylometric.csv")
bec_modernbert = pd.read_csv(DATA_PROCESSED / "bec_modernbert.csv")

print(f"BEC raw: {len(bec_raw)}")
print(f"BEC stylometric: {len(bec_stylo)}")
print(f"BEC ModernBERT: {len(bec_modernbert)}")

# Align lengths (in case of any mismatch)
min_len = min(len(bec_stylo), len(bec_modernbert))
bec_stylo = bec_stylo.iloc[:min_len].reset_index(drop=True)
bec_modernbert = bec_modernbert.iloc[:min_len].reset_index(drop=True)

# All BEC emails are phishing - we treat them as "should be flagged as phishing"
# True label: these are AI-generated phishing = class 2
# But success = flagged as ANY phishing (class 1 or 2), failure = called legitimate (class 0)

print("EXTERNAL GENERALISATION TEST — Unseen Gemini BEC Phishing")

# System 1 — ModernBERT alone
mb_flagged = (bec_modernbert['llm_label'] != 0).sum()
print(f"\n1. ModernBERT alone:")
print(f"   Flagged as phishing: {mb_flagged}/{min_len} ({mb_flagged/min_len*100:.1f}%)")

# System 2 — Stylometric only
with open(MODELS_DIR / "baseline_stylo_only.pkl", 'rb') as f:
    stylo_model = pickle.load(f)
# Match feature columns
stylo_cols = stylo_features.columns
bec_stylo_aligned = bec_stylo[stylo_cols]
stylo_preds = stylo_model.predict(bec_stylo_aligned)
stylo_flagged = (stylo_preds != 0).sum()
print(f"\n2. Stylometric only:")
print(f"   Flagged as phishing: {stylo_flagged}/{min_len} ({stylo_flagged/min_len*100:.1f}%)")

# System 3 — Fusion (ModernBERT + Stylometric)
with open(MODELS_DIR / "xgboost_modernbert.pkl", 'rb') as f:
    fusion_model = pickle.load(f)
bec_fusion = pd.concat([
    bec_stylo_aligned.reset_index(drop=True),
    bec_modernbert[['conf_legitimate','conf_human_phishing','conf_ai_phishing']].reset_index(drop=True)
], axis=1)
fusion_preds = fusion_model.predict(bec_fusion)
fusion_flagged = (fusion_preds != 0).sum()
print(f"\n3. Fusion (ModernBERT + Stylometric):")
print(f"   Flagged as phishing: {fusion_flagged}/{min_len} ({fusion_flagged/min_len*100:.1f}%)")

print("SUMMARY — Detection Rate on Unseen AI Phishing")
print(f"  ModernBERT alone:    {mb_flagged/min_len*100:.1f}%")
print(f"  Stylometric only:    {stylo_flagged/min_len*100:.1f}%")
print(f"  Fusion system:       {fusion_flagged/min_len*100:.1f}%")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             roc_curve, auc)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Load all features
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
modernbert = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")
stylo_diverse = pd.read_csv(DATA_PROCESSED / "stylometric_features_diverse.csv")
modernbert_diverse = pd.read_csv(DATA_PROCESSED / "modernbert_diverse_features.csv")

# Load models
with open(MODELS_DIR / "xgboost_modernbert.pkl", 'rb') as f:
    model_original = pickle.load(f)
with open(MODELS_DIR / "xgboost_diverse.pkl", 'rb') as f:
    model_diverse = pickle.load(f)

# Build feature matrices
X_original = pd.concat([
    stylo.drop(columns=['label']),
    modernbert[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1)
y_original = stylo['label']

X_diverse = pd.concat([
    stylo_diverse.drop(columns=['label']),
    modernbert_diverse[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1)
y_diverse = stylo_diverse['label']

# Same splits
_, X_temp, _, y_temp = train_test_split(
    X_original, y_original, test_size=0.30, random_state=42, stratify=y_original)
_, X_test_orig, _, y_test_orig = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

_, X_temp_d, _, y_temp_d = train_test_split(
    X_diverse, y_diverse, test_size=0.30, random_state=42, stratify=y_diverse)
_, X_test_div, _, y_test_div = train_test_split(
    X_temp_d, y_temp_d, test_size=0.50, random_state=42, stratify=y_temp_d)

# Get predictions
preds_orig = model_original.predict(X_test_orig)
proba_orig = model_original.predict_proba(X_test_orig)
preds_div = model_diverse.predict(X_test_div)
proba_div = model_diverse.predict_proba(X_test_div)

print("Setup complete — ready to generate graphs")
print(f"Original test set: {len(y_test_orig)} emails")
print(f"Diverse test set: {len(y_test_div)} emails")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Legitimate', 'Human\nPhishing', 'AI\nPhishing']

# Original system
cm_orig = confusion_matrix(y_test_orig, preds_orig)
sns.heatmap(cm_orig, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0],
            linewidths=0.5, linecolor='white')
axes[0].set_title('Original Fusion System\n(ModernBERT + Stylometric)',
                  fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)

# Diverse system
cm_div = confusion_matrix(y_test_div, preds_div)
sns.heatmap(cm_div, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=axes[1],
            linewidths=0.5, linecolor='white')
axes[1].set_title('Diverse Fusion System\n(ModernBERT + Stylometric + BEC Training)',
                  fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_xlabel('Predicted Label', fontsize=11)

plt.suptitle('Confusion Matrices — Fusion Classifier Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")

In [ ]:
#Macro F1 Comparison Across All Systems
systems = [
    'Zero-shot\nQwen2.5',
    'Stylometric\nOnly',
    'TF-IDF +\nLog. Reg.',
    'TF-IDF +\nRandom Forest',
    'ModernBERT\nStandalone',
    'Original\nFusion',
    'Diverse\nFusion'
]
f1_scores = [0.5121, 0.9600, 0.9759, 0.9703, 0.9911, 0.9900, 0.9879]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#f1c40f',
          '#2ecc71', '#3498db', '#2E86AB']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(systems, f1_scores, color=colors,
              edgecolor='white', linewidth=1.5, width=0.6)

for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_ylim(0.4, 1.05)
ax.set_ylabel('Macro F1 Score', fontsize=12)
ax.set_title('Macro F1 Comparison Across All Systems',
             fontsize=14, fontweight='bold')
ax.axhline(y=0.95, color='gray', linestyle='--',
           alpha=0.5, label='0.95 threshold')
ax.axhline(y=0.99, color='green', linestyle='--',
           alpha=0.5, label='0.99 threshold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: macro_f1_comparison.png")

In [ ]:
#BEC Detection Progression
stages = ['Original\nFusion', 'Cleaned\nFusion',
          'Diverse\nModernBERT\nAlone', 'Diverse\nFusion']
detection = [2.7, 6.2, 99.7, 99.8]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#2E86AB']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(stages, detection, color=colors,
              edgecolor='white', linewidth=1.5, width=0.5)

for bar, val in zip(bars, detection):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val}%', ha='center', va='bottom',
            fontsize=12, fontweight='bold')

ax.set_ylim(0, 115)
ax.set_ylabel('Detection Rate (%)', fontsize=12)
ax.set_title('BEC Phishing Detection Rate — System Progression',
             fontsize=14, fontweight='bold')
ax.annotate('Diverse training\ndata added here',
            xy=(2, 99.7), xytext=(1.2, 75),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, color='black')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "bec_detection_progression.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: bec_detection_progression.png")

In [ ]:
#ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ['Legitimate', 'Human Phishing', 'AI Phishing']
colors_roc = ['#3498db', '#e74c3c', '#2ecc71']

for ax, (preds, proba, y_test, title) in zip(axes, [
    (preds_orig, proba_orig, y_test_orig, 'Original Fusion System'),
    (preds_div, proba_div, y_test_div, 'Diverse Fusion System')
]):
    y_bin = label_binarize(y_test, classes=[0, 1, 2])
    for i, (cls_name, color) in enumerate(zip(class_names, colors_roc)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{cls_name} (AUC = {roc_auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'ROC Curves — {title}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_curves.png")

In [ ]:
#Feature Importance
feature_names = list(X_original.columns)
importances = model_original.feature_importances_
indices = np.argsort(importances)[::-1][:20]
top_features = [feature_names[i] for i in indices]
top_importances = importances[indices]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(20), top_importances[::-1],
               color='#2E86AB', edgecolor='white', linewidth=0.5)
ax.set_yticks(range(20))
ax.set_yticklabels(top_features[::-1], fontsize=10)
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Top 20 Most Important Features\nXGBoost Fusion Classifier',
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

In [ ]:
#Metrics Heatmap
metrics_data = {
    'System': ['Zero-shot', 'Stylometric Only', 'TF-IDF+LR',
                'TF-IDF+RF', 'ModernBERT', 'Original Fusion', 'Diverse Fusion'],
    'Macro F1': [0.5121, 0.9600, 0.9759, 0.9703, 0.9911, 0.9900, 0.9879],
    'AUC-ROC': [0.0, 0.9952, 0.9950, 0.9940, 0.9993, 0.9980, 0.9985],
    'MCC': [0.5073, 0.9302, 0.9450, 0.9380, 0.9844, 0.9825, 0.9820],
    'AI Recall': [0.005, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000],
    'FPR': [0.120, 0.077, 0.063, 0.068, 0.015, 0.017, 0.018]
}

metrics_df = pd.DataFrame(metrics_data).set_index('System')

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, linecolor='white',
            vmin=0, vmax=1, ax=ax)
ax.set_title('Evaluation Metrics Heatmap — All Systems',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('System', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "metrics_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: metrics_heatmap.png")

In [ ]:
# Training Curve
epochs = [1, 2, 3, 4]
train_loss = [0.1327, 0.0434, 0.0153, 0.0001]
val_loss = [0.0593, 0.0636, 0.0618, 0.0660]
val_f1 = [0.9807, 0.9867, 0.9884, 0.9878]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

line1 = ax1.plot(epochs, train_loss, 'b-o', linewidth=2,
                  markersize=8, label='Training Loss')
line2 = ax1.plot(epochs, val_loss, 'r-o', linewidth=2,
                  markersize=8, label='Validation Loss')
line3 = ax2.plot(epochs, val_f1, 'g-s', linewidth=2,
                  markersize=8, label='Validation Macro F1')

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12, color='black')
ax2.set_ylabel('Macro F1 Score', fontsize=12, color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.set_ylim(0.95, 1.0)

ax1.set_title('ModernBERT Training Curve — Diverse Dataset',
              fontsize=14, fontweight='bold')
ax1.set_xticks(epochs)

lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=10)
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curve.png")

In [ ]:
# Scenario 1: Different Train/Test Split Ratios

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, roc_auc_score, 
                             matthews_corrcoef, confusion_matrix,
                             recall_score)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import time
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load diverse features
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_diverse.csv")
modernbert = pd.read_csv(DATA_PROCESSED / "modernbert_diverse_features.csv")

X = pd.concat([
    stylo.drop(columns=['label']),
    modernbert[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1)
y = stylo['label']

print(f"Feature matrix: {X.shape}")
print(f"Class distribution:\n{y.value_counts().sort_index()}")

# Three split ratios to test
splits = [
    {"name": "70/15/15", "test_size": 0.30, "val_ratio": 0.50},
    {"name": "80/10/10", "test_size": 0.20, "val_ratio": 0.50},
    {"name": "60/20/20", "test_size": 0.40, "val_ratio": 0.50},
]

def train_xgb(X_train, y_train):
    cc = y_train.value_counts().sort_index()
    total = len(y_train)
    cw = {cls: total/(len(cc)*count) for cls, count in cc.items()}
    sw = y_train.map(cw)
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='mlogloss', verbosity=0)
    model.fit(X_train, y_train, sample_weight=sw)
    return model

def compute_metrics(y_true, y_pred, y_proba, model, X_test):
    # AUC-ROC
    y_bin = label_binarize(y_true, classes=[0,1,2])
    auc = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
    
    # FPR for legitimate class
    cm = confusion_matrix(y_true, y_pred)
    fp_legit = cm[0][1] + cm[0][2]
    total_legit = cm[0].sum()
    fpr = fp_legit / total_legit if total_legit > 0 else 0
    
    # AI phishing recall
    ai_recall = recall_score(y_true, y_pred, labels=[2], average='macro')
    
    # Inference time
    start = time.time()
    _ = model.predict(X_test)
    inference_time = (time.time() - start) / len(X_test) * 1000
    
    return {
        'Macro F1': round(f1_score(y_true, y_pred, average='macro'), 4),
        'AUC-ROC': round(auc, 4),
        'MCC': round(matthews_corrcoef(y_true, y_pred), 4),
        'FPR': round(fpr, 4),
        'AI Recall': round(ai_recall, 4),
        'Inference Time (ms)': round(inference_time, 4)
    }

# Run all three scenarios
results = []
print("\nRunning Scenario 1 — Split Ratio Comparison\n")

for split in splits:
    print(f"Testing split: {split['name']}")
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=split['test_size'],
        random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=split['val_ratio'],
        random_state=42, stratify=y_temp)
    
    print(f"  Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")
    
    model = train_xgb(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)
    
    metrics = compute_metrics(y_test, preds, proba, model, X_test)
    metrics['Split'] = split['name']
    metrics['Train Size'] = len(y_train)
    metrics['Test Size'] = len(y_test)
    results.append(metrics)
    
    print(f"  Macro F1: {metrics['Macro F1']}")
    print(f"  AI Recall: {metrics['AI Recall']}")
    print()

# Display results table
results_df = pd.DataFrame(results).set_index('Split')
print("\nSCENARIO 1 RESULTS ")
print(results_df.to_string())

# Save
results_df.to_csv(BASE_DIR / "results" / "scenario1_split_ratios.csv")
print("\nSaved: scenario1_split_ratios.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = BASE_DIR / "results"

splits = ['70/15/15', '80/10/10', '60/20/20']
metrics = ['Macro F1', 'AUC-ROC', 'MCC', 'AI Recall']
values = {
    'Macro F1':  [0.9879, 0.9901, 0.9889],
    'AUC-ROC':   [0.9980, 0.9993, 0.9968],
    'MCC':       [0.9825, 0.9857, 0.9839],
    'AI Recall': [0.9971, 1.0000, 0.9989],
}

x = np.arange(len(splits))
width = 0.2
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

fig, ax = plt.subplots(figsize=(11, 6))

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, values[metric], width,
                  label=metric, color=color,
                  edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, values[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom',
                fontsize=7.5, fontweight='bold')

ax.set_ylim(0.97, 1.01)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(splits, fontsize=11)
ax.set_xlabel('Train/Test Split Ratio', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scenario 1 — Performance Across Different Train/Test Split Ratios',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "scenario1_split_ratios.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scenario1_split_ratios.png")

In [ ]:
# Scenario 2 - Ablation Study

print("Running Scenario 2 — Ablation Study\n")

# Reset indices to avoid alignment issues
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

# Use standard 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X_reset, y_reset, test_size=0.30, random_state=42, stratify=y_reset)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

stylo_cols = stylo.drop(columns=['label']).columns
mb_cols = ['conf_legitimate','conf_human_phishing','conf_ai_phishing']

# Load zero-shot features — only 10,492 rows
# Use original dataset split for zero-shot comparison
stylo_orig = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
modernbert_orig = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")
llm = pd.read_csv(DATA_PROCESSED / "llm_features.csv")

X_orig = pd.concat([
    stylo_orig.drop(columns=['label']),
    llm[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1).reset_index(drop=True)
y_orig = stylo_orig['label'].reset_index(drop=True)

X_orig_train, X_orig_temp, y_orig_train, y_orig_temp = train_test_split(
    X_orig, y_orig, test_size=0.30, random_state=42, stratify=y_orig)
X_orig_val, X_orig_test, y_orig_val, y_orig_test = train_test_split(
    X_orig_temp, y_orig_temp, test_size=0.50, random_state=42, stratify=y_orig_temp)

ablation_results = []

configs = [
    {'name': 'Stylometric only',
     'X_train': X_train[stylo_cols],
     'X_test': X_test[stylo_cols],
     'y_train': y_train, 'y_test': y_test},
    {'name': 'ModernBERT only',
     'X_train': X_train[mb_cols],
     'X_test': X_test[mb_cols],
     'y_train': y_train, 'y_test': y_test},
    {'name': 'Zero-shot + XGBoost',
     'X_train': X_orig_train[['conf_legitimate',
                               'conf_human_phishing',
                               'conf_ai_phishing']],
     'X_test': X_orig_test[['conf_legitimate',
                             'conf_human_phishing',
                             'conf_ai_phishing']],
     'y_train': y_orig_train, 'y_test': y_orig_test},
    {'name': 'Full Fusion (Diverse)',
     'X_train': X_train,
     'X_test': X_test,
     'y_train': y_train, 'y_test': y_test},
]

for config in configs:
    print(f"Testing: {config['name']}")
    model = train_xgb(config['X_train'], config['y_train'])
    preds = model.predict(config['X_test'])
    proba = model.predict_proba(config['X_test'])
    metrics = compute_metrics(
        config['y_test'], preds, proba, model, config['X_test'])
    metrics['Configuration'] = config['name']
    ablation_results.append(metrics)
    print(f"  Macro F1: {metrics['Macro F1']}")
    print(f"  AI Recall: {metrics['AI Recall']}\n")

ablation_df = pd.DataFrame(ablation_results).set_index('Configuration')
print("\n SCENARIO 2 RESULTS ")
print(ablation_df.to_string())

ablation_df.to_csv(BASE_DIR / "results" / "scenario2_ablation.csv")
print("\nSaved: scenario2_ablation.csv")

In [ ]:
configs = ['Zero-shot\n+XGBoost', 'Stylometric\nOnly', 
           'ModernBERT\nOnly', 'Full Fusion\n(Diverse)']
metrics_s2 = {
    'Macro F1':  [0.6247, 0.9445, 0.9868, 0.9879],
    'AUC-ROC':   [0.7883, 0.9943, 0.9982, 0.9980],
    'MCC':       [0.4934, 0.9200, 0.9809, 0.9825],
    'AI Recall': [0.6551, 0.9826, 0.9956, 0.9971],
}

x = np.arange(len(configs))
width = 0.2
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, color) in enumerate(zip(metrics_s2.keys(), colors)):
    bars = ax.bar(x + i*width, metrics_s2[metric], width,
                  label=metric, color=color,
                  edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, metrics_s2[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=7.5, fontweight='bold')

ax.set_ylim(0.4, 1.05)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(configs, fontsize=10)
ax.set_xlabel('System Configuration', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scenario 2 — Ablation Study: Component Contribution Analysis',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "scenario2_ablation.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scenario2_ablation.png")

In [ ]:
print("Running Scenario 3 — Dataset Composition\n")

# Load all three dataset configurations
# Config 1 — Original single source (Qwen2.5 only)
stylo_orig = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
mb_orig = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")

X_orig = pd.concat([
    stylo_orig.drop(columns=['label']),
    mb_orig[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1).reset_index(drop=True)
y_orig = stylo_orig['label'].reset_index(drop=True)

# Config 2 — Cleaned single source (salutations removed)
stylo_clean = pd.read_csv(DATA_PROCESSED / "stylometric_features_cleaned.csv")
mb_clean = pd.read_csv(DATA_PROCESSED / "modernbert_cleaned_features.csv")

X_clean = pd.concat([
    stylo_clean.drop(columns=['label']),
    mb_clean[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1).reset_index(drop=True)
y_clean = stylo_clean['label'].reset_index(drop=True)

# Config 3 — Diverse multi-source (Qwen2.5 + Gemini BEC)
stylo_div = pd.read_csv(DATA_PROCESSED / "stylometric_features_diverse.csv")
mb_div = pd.read_csv(DATA_PROCESSED / "modernbert_diverse_features.csv")

X_div = pd.concat([
    stylo_div.drop(columns=['label']),
    mb_div[['conf_legitimate','conf_human_phishing','conf_ai_phishing']]
], axis=1).reset_index(drop=True)
y_div = stylo_div['label'].reset_index(drop=True)

print(f"Original dataset:  {len(X_orig)} emails")
print(f"Cleaned dataset:   {len(X_clean)} emails")
print(f"Diverse dataset:   {len(X_div)} emails")

configs = [
    {'name': 'Original\n(Qwen2.5 only)', 'X': X_orig, 'y': y_orig},
    {'name': 'Cleaned\n(Salutations removed)', 'X': X_clean, 'y': y_clean},
    {'name': 'Diverse\n(Qwen2.5 + Gemini)', 'X': X_div, 'y': y_div},
]

scenario3_results = []

for config in configs:
    print(f"\nTesting: {config['name'].replace(chr(10), ' ')}")

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        config['X'], config['y'],
        test_size=0.30, random_state=42, stratify=config['y'])
    _, X_te, _, y_te = train_test_split(
        X_tmp, y_tmp,
        test_size=0.50, random_state=42, stratify=y_tmp)

    model = train_xgb(X_tr, y_tr)
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)

    metrics = compute_metrics(y_te, preds, proba, model, X_te)
    metrics['Configuration'] = config['name'].replace('\n', ' ')
    metrics['Dataset Size'] = len(config['X'])
    scenario3_results.append(metrics)

    print(f"  Macro F1:  {metrics['Macro F1']}")
    print(f"  AI Recall: {metrics['AI Recall']}")
    print(f"  FPR:       {metrics['FPR']}")

s3_df = pd.DataFrame(scenario3_results).set_index('Configuration')
print("\n SCENARIO 3 RESULTS ")
print(s3_df.to_string())

s3_df.to_csv(BASE_DIR / "results" / "scenario3_dataset_composition.csv")
print("\nSaved: scenario3_dataset_composition.csv")

In [ ]:
configs_s3 = ['Original\n(Qwen2.5 only)',
              'Cleaned\n(Salutations removed)',
              'Diverse\n(Qwen2.5 + Gemini)']

metrics_s3 = {
    'Macro F1':  [0.9900, 0.9900, 0.9879],
    'AUC-ROC':   [0.9980, 0.9986, 0.9980],
    'MCC':       [0.9825, 0.9825, 0.9825],
    'AI Recall': [1.0000, 1.0000, 0.9971],
}

x = np.arange(len(configs_s3))
width = 0.2
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, color) in enumerate(zip(metrics_s3.keys(), colors)):
    bars = ax.bar(x + i*width, metrics_s3[metric], width,
                  label=metric, color=color,
                  edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, metrics_s3[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.0002,
                f'{val:.4f}', ha='center', va='bottom',
                fontsize=7.5, fontweight='bold')

ax.set_ylim(0.97, 1.01)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(configs_s3, fontsize=10)
ax.set_xlabel('Dataset Configuration', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scenario 3 — Performance Across Dataset Compositions',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "scenario3_dataset_composition.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scenario3_dataset_composition.png")

In [ ]:
import seaborn as sns

# Combined heatmap of all three scenarios
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scenario 1 heatmap
s1_data = pd.DataFrame({
    'Macro F1':  [0.9879, 0.9901, 0.9889],
    'AUC-ROC':   [0.9980, 0.9993, 0.9968],
    'MCC':       [0.9825, 0.9857, 0.9839],
    'FPR':       [0.0117, 0.0100, 0.0150],
    'AI Recall': [0.9971, 1.0000, 0.9989],
}, index=['70/15/15', '80/10/10', '60/20/20'])

sns.heatmap(s1_data, annot=True, fmt='.4f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Scenario 1\nSplit Ratios',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('')

# Scenario 2 heatmap
s2_data = pd.DataFrame({
    'Macro F1':  [0.6247, 0.9445, 0.9868, 0.9879],
    'AUC-ROC':   [0.7883, 0.9943, 0.9982, 0.9980],
    'MCC':       [0.4934, 0.9200, 0.9809, 0.9825],
    'FPR':       [0.1217, 0.1083, 0.0150, 0.0117],
    'AI Recall': [0.6551, 0.9826, 0.9956, 0.9971],
}, index=['Zero-shot', 'Stylometric', 'ModernBERT', 'Full Fusion'])

sns.heatmap(s2_data, annot=True, fmt='.4f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Scenario 2\nAblation Study',
                  fontsize=11, fontweight='bold')
axes[1].set_ylabel('')

# Scenario 3 heatmap
s3_data = pd.DataFrame({
    'Macro F1':  [0.9900, 0.9900, 0.9879],
    'AUC-ROC':   [0.9980, 0.9986, 0.9980],
    'MCC':       [0.9825, 0.9825, 0.9825],
    'FPR':       [0.0167, 0.0133, 0.0117],
    'AI Recall': [1.0000, 1.0000, 0.9971],
}, index=['Original', 'Cleaned', 'Diverse'])

sns.heatmap(s3_data, annot=True, fmt='.4f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[2], vmin=0, vmax=1)
axes[2].set_title('Scenario 3\nDataset Composition',
                  fontsize=11, fontweight='bold')
axes[2].set_ylabel('')

plt.suptitle('All Scenarios — Evaluation Metrics Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "all_scenarios_heatmap.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: all_scenarios_heatmap.png")

In [ ]:
# benchmark comparison

import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path("C:/phishing_detection")
RESULTS_DIR = BASE_DIR / "results"

RESULTS_DIR = BASE_DIR / "results"

systems = [
    'Heiding et al.\n(IEEE 2024)',
    'Zhang et al.\n(IEEE 2025)',
    'Afane et al.\n(IEEE BD 2024)',
    'Chanis &\nArampatzis\n(ACM 2024)',
    'Our System\n(This Work)'
]

# Using actual reported values
# Heiding: >0.95 accuracy — using 0.95 as conservative estimate
# Zhang: 0.9756 F1
# Afane: recall 0.9895 — using as proxy
# Chanis: 0.9843 F1
# Ours: 0.9879 Macro F1
values = [0.9500, 0.9756, 0.9895, 0.9843, 0.9879]
colors = ['#888888', '#888888', '#888888', '#888888', '#2E86AB']
hatches = ['', '', '', '', '']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(systems, values, color=colors,
              edgecolor='white', linewidth=1.5, width=0.5)

for bar, val, sys in zip(bars, values, systems):
    label = f'{val:.4f}'
    if 'Heiding' in sys:
        label = '>0.9500\n(accuracy)'
    elif 'Afane' in sys:
        label = '0.9895\n(recall)'
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            label, ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_ylim(0.90, 1.02)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Benchmark Comparison — Our System vs Published Literature\n(Note: metrics differ across papers due to inconsistent reporting)',
             fontsize=12, fontweight='bold')
ax.axhline(y=0.9843, color='gray', linestyle='--',
           alpha=0.5, label='Best published F1 (Chanis 2024: 0.9843)')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "benchmark_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: benchmark_comparison.png")